# EEGConformer — BCI-IV-2a (Neuro-Fabric)

Trains and exports the production `eegconformer.pt` → `eegconformer.onnx` artefact.

**Runtime:** GPU (T4 is enough). **Wall clock:** ~30–60 min.
**Tested on:** Google Colab, Python 3.10/3.11/3.12, CUDA 12.


## 1. Clone Neuro-Fabric


In [ ]:
import os, subprocess
REPO   = os.environ.get('NEUROFABRIC_REPO',   'https://github.com/ouadaououhoussemdroid/neuro-fabric-core.git')
BRANCH = os.environ.get('NEUROFABRIC_BRANCH', 'main')
if not os.path.isdir('/content/repo'):
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/repo'])
%cd /content/repo/training


## 2. Install pinned dependencies

We pin the **exact** versions that are known to be compatible:
Braindecode ↔ MOABB ↔ MNE ↔ PyTorch. See `docs/audits/training-dependency-audit.md`.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -q -r requirements.txt


## 3. Automated compatibility check

Verifies installed versions match the pinned contract and that the dataset
alias `BNCI2014_001` resolves before any training runs.


In [ ]:
!python scripts/check_compat.py


## 4. Verify GPU


In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 5. Review the contract


In [ ]:
!cat configs/eegconformer-bciiv2a.yaml


## 6. Acquire BCI-IV-2a (MOABB)


In [ ]:
!python scripts/acquire_dataset.py


## 7. Preprocess → (22, 1000) @ 250 Hz, z-scored


In [ ]:
!python scripts/preprocess.py


## 8. Train EEGConformer


In [ ]:
!python scripts/train.py


## 9. Validate (cross-subject hold-out)


In [ ]:
!python scripts/validate.py


## 10. Evaluate embeddings (cosine recall@10)


In [ ]:
!python scripts/evaluate.py


## 11. Export to ONNX (opset 17, parity ≥ 0.999)


In [ ]:
!python scripts/export_onnx.py


## 12. Package artefact bundle


In [ ]:
!python scripts/package.py
!ls -la artefacts/


## 13. Download artefacts to local machine


In [ ]:
from google.colab import files
import glob, os
for p in sorted(glob.glob('artefacts/**/*.onnx', recursive=True) +
                glob.glob('artefacts/**/*.pt',   recursive=True) +
                glob.glob('artefacts/**/*.zip',  recursive=True)):
    print('downloading', p, os.path.getsize(p), 'bytes')
    files.download(p)
